In [1]:
import numpy as np
import pandas as pd
import math

## 1-1 csv 파일 읽기



In [2]:
home = '/content/1.01.house.price.csv'
df_house = pd.read_csv(home)
df_house

,ID,size,year,price
0,1,34,5,5.0
1,2,25,5,2.5
2,3,30,2,4.0
3,4,38,20,3.0
4,5,44,12,3.3
5,6,48,18,4.2
6,7,52,22,4.6
7,8,60,19,6.0
8,9,34,18,3.0
9,10,34,22,2.9


In [3]:
# 첫 번째 행, 전체 데이터 행 수 출력
print(len(df_house))
df_house.head(1)
print("첫 번째 행 price 자료형:", type(df_house['price'].iloc[0]))

10
첫 번째 행 price 자료형: <class 'numpy.float64'>


In [4]:
# 비정상 출력
home = '/content.csv'

try:
    df_house = pd.read_csv(home)

except FileNotFoundError:
    print('파일명과 업로드 여부를 확인하세요')

else:
    df_house = pd.read_csv(home)
    print(home)


파일명과 업로드 여부를 확인하세요


In [5]:
# 정상 출력

home = '/content/1.01.house.price.csv'

try:
    df_house = pd.read_csv(home)

except FileNotFoundError:
    print('파일명과 업로드 여부를 확인하세요')

else:
    df_house = pd.read_csv(home)
    print(home)


/content/1.01.house.price.csv


## 1-2 문자열을 숫자로 변환하기

In [6]:
# 빈 리스트 생성
houses = []
excluded = 0   # 제외된 행 수

# 반복문으로 각 행 순회
for idx, row in df_house.iterrows():
    try:
        # 필수 값 누락 확인
        for col in ['ID', 'size', 'year', 'price']:
            value = row[col]
            if pd.isna(value) or str(value).strip() == '':
                raise ValueError(f"필수 값 누락 ({col})")

        # 자료형 변환
        house = {
            'ID': int(row['ID']),
            'size': int(row['size']),
            'year': int(row['year']),
            'price': float(row['price'])
        }

    except (ValueError, TypeError) as e:
        # 제외 행의 위치와 사유 출력, 반복문은 계속 진행
        excluded += 1
        print(f"행 위치 {idx}: 제외 - {e}")

    else:
        houses.append(house)

# 정상 저장 행 수와 제외 행 수 출력
print(f"정상 저장 행 수: {len(houses)}")
print(f"제외 행 수: {excluded}")
print(houses[:2])

정상 저장 행 수: 10
제외 행 수: 0
[{'ID': 1, 'size': 34, 'year': 5, 'price': 5.0}, {'ID': 2, 'size': 25, 'year': 5, 'price': 2.5}]


## 1-3 조건에 맞는 주택 찾기

In [7]:
# 원본 유지 / houses_test 복사
houses_test = houses.copy()

houses_test.append({"ID": 101, "year": 5, "price": 3})  # size 키 없음
houses_test.append({"ID": 102, "size": 35, "year": 5, "price": None})  # 값 없음
houses_test.append({"ID": 103, "size": "오류", "year": 5, "price": 3})  # 숫자로 변환할 수 없음
houses_test.append({"ID": 104, "size": 35, "year": 5, "price": 3})  # 정상 데이터

houses_test

[{'ID': 1, 'size': 34, 'year': 5, 'price': 5.0},
 {'ID': 2, 'size': 25, 'year': 5, 'price': 2.5},
 {'ID': 3, 'size': 30, 'year': 2, 'price': 4.0},
 {'ID': 4, 'size': 38, 'year': 20, 'price': 3.0},
 {'ID': 5, 'size': 44, 'year': 12, 'price': 3.3},
 {'ID': 6, 'size': 48, 'year': 18, 'price': 4.2},
 {'ID': 7, 'size': 52, 'year': 22, 'price': 4.6},
 {'ID': 8, 'size': 60, 'year': 19, 'price': 6.0},
 {'ID': 9, 'size': 34, 'year': 18, 'price': 3.0},
 {'ID': 10, 'size': 34, 'year': 22, 'price': 2.9},
 {'ID': 101, 'year': 5, 'price': 3},
 {'ID': 102, 'size': 35, 'year': 5, 'price': None},
 {'ID': 103, 'size': '오류', 'year': 5, 'price': 3},
 {'ID': 104, 'size': 35, 'year': 5, 'price': 3}]

In [8]:
selected = []   # ④ 조건을 만족하는 주택을 저장할 리스트

for house in houses_test:
    hid = house.get('ID')

    # 1) size, price 키 존재 확인
    missing = [k for k in ('size', 'price') if k not in house]
    if missing:
        print(f"ID {hid}: 제외 - 키 누락 ({', '.join(missing)})")
        continue

    # 2) None·결측값·빈 문자열 확인
    empty = [k for k in ('size', 'price')
             if house[k] is None or pd.isna(house[k]) or str(house[k]).strip() == '']
    if empty:
        print(f"ID {hid}: 제외 - None 또는 결측값 ({', '.join(empty)})")
        continue

    # 3) 숫자 변환 (숫자 형태의 문자열은 변환해서 검사)
    try:
        size = float(house['size'])
        price = float(house['price'])
        if not (math.isfinite(size) and math.isfinite(price)):
            raise ValueError("유한한 숫자가 아님")
    except (ValueError, TypeError):
        print(f"ID {hid}: 제외 - 숫자로 변환 불가 "
              f"(size={house['size']!r}, price={house['price']!r})")
        continue

    # ②③ 유효하지 않은 값 제외: size가 0 이하 or price가 음수
    if size <= 0 or price < 0:
        print(f"ID {hid}: 제외 - 유효하지 않은 값 (size={size}, price={price})")
        continue

    # ④ 두 조건을 모두 만족하면 selected에 저장
    if size >= 30 and price <= 4:
        selected.append(house)

# ⑥ 선택한 주택의 ID를 하나씩 출력
print("선택된 주택 ID:")
for house in selected:
    print(house['ID'])

# ⑦ 선택한 주택의 전체 건수 출력
print(f"선택된 주택 수: {len(selected)}")

ID 101: 제외 - 키 누락 (size)
ID 102: 제외 - None 또는 결측값 (price)
ID 103: 제외 - 숫자로 변환 불가 (size='오류', price=3)
선택된 주택 ID:
3
4
5
9
10
104
선택된 주택 수: 6


## 1-4 함수와 예외 처리

In [9]:
# 가격을 크기로 나눈 값을 계산
def unit_price(price, size):
    """가격 ÷ 크기를 반환합니다. 잘못된 입력이면 안내 문구를 출력하고 None을 반환합니다."""
    try:
        # ② 숫자 또는 숫자 형태의 문자열을 float으로 변환
        price = float(price)
        size = float(size)

        # "inf", "nan"처럼 변환은 되지만 계산에 쓸 수 없는 값 차단
        if not (math.isfinite(price) and math.isfinite(size)):
            raise ValueError("유한한 숫자가 아님")

    except ValueError:
        # ⑤ 숫자로 변환할 수 없는 문자열
        print("값 오류: 숫자로 변환할 수 없는 문자열입니다.")
        return None
    except TypeError:
        # ⑥ None 등 숫자로 변환할 수 없는 자료형
        print("자료형 오류: 숫자 또는 숫자 형태의 문자열을 입력하세요.")
        return None

    # ④ 크기가 0 이하이거나 가격이 음수이면 원인별 안내 문구 출력 후 None 반환
    #    (나눗셈 전에 검사해서 0으로 나누기 방지)
    if size <= 0:
        print(f"크기 오류: 크기는 0보다 커야 합니다. (size={size})")
        return None
    if price < 0:
        print(f"가격 오류: 가격은 음수일 수 없습니다. (price={price})")
        return None

    # ③ 정상 입력이면 가격 ÷ 크기 반환
    return price / size


# ⑦ 잘못된 입력이 있어도 None을 반환하므로 반복문이 중단되지 않음
# ⑧ 전체 houses에 적용: 주택 ID와 크기당 가격 출력
for house in houses:
    result = unit_price(house['price'], house['size'])

    if result is None:   # 오류 안내는 함수 안에서 이미 출력됨
        continue

    print(f"ID {house['ID']}: 크기당 가격 = {result:.4f}")

ID 1: 크기당 가격 = 0.1471
ID 2: 크기당 가격 = 0.1000
ID 3: 크기당 가격 = 0.1333
ID 4: 크기당 가격 = 0.0789
ID 5: 크기당 가격 = 0.0750
ID 6: 크기당 가격 = 0.0875
ID 7: 크기당 가격 = 0.0885
ID 8: 크기당 가격 = 0.1000
ID 9: 크기당 가격 = 0.0882
ID 10: 크기당 가격 = 0.0853


In [10]:
print(unit_price(4, 40))
print(unit_price("4", "40"))
print(unit_price(4, 0))
print(unit_price(-4, 40))
print(unit_price("오류", 40))
print(unit_price(None, 40))
print(unit_price(6, 30))

0.1
0.1
크기 오류: 크기는 0보다 커야 합니다. (size=0.0)
None
가격 오류: 가격은 음수일 수 없습니다. (price=-4.0)
None
값 오류: 숫자로 변환할 수 없는 문자열입니다.
None
자료형 오류: 숫자 또는 숫자 형태의 문자열을 입력하세요.
None
0.2


## 1-5 목적에 맞는 자료구조 선택하기

In [11]:
# ① 전체 주택 ID를 원래 순서대로 저장 → 리스트(list)
# [선택 이유] 순서 유지: 데이터가 들어온 순서(인덱스) 그대로 저장됨.
#            중복 허용: 같은 ID가 있어도 제거하지 않고 전부 저장함.
# [선택하지 않은 자료구조: 집합(set)]
#   - 집합은 순서를 유지하지 않고(출력 순서 보장 없음), 중복을 자동 제거함.
#   - "원래 순서대로 전체 ID"라는 목적에는 순서·전체 보존이 안 되어 맞지 않음.
house_ids = [house['ID'] for house in houses]
print("전체 주택 ID:", house_ids)

# ④ 최소 크기 30, 최대 가격 4를 하나의 묶음으로 저장 → 튜플(tuple)
# [선택 이유] 변경 불가: 생성 후 원소 추가·삭제·교체가 불가능해서
#            검색 기준값이 실수로 바뀌는 것을 막을 수 있음.
#            두 값(최소 크기, 최대 가격)을 하나의 묶음으로 저장함.
# [선택하지 않은 자료구조: 리스트(list)]
#   - 리스트는 변경 가능해서 append, remove, 값 교체가 모두 가능함.
#   - "수정할 수 없어야 한다"는 요구에 맞지 않음.
criteria = (30, 4)   # (최소 크기, 최대 가격)
print("검색 조건:", criteria)

전체 주택 ID: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
검색 조건: (30, 4)


In [12]:
# ② year 값을 중복 없이 저장 → 집합(set)
# [선택 이유] 중복 불허: 같은 값을 여러 번 넣어도 하나만 남음.
# [선택하지 않은 자료구조: 리스트(list)]
#   - 리스트는 중복을 허용해서 같은 year가 여러 번 저장됨.
#   - 중복을 없애려면 별도의 검사나 변환이 필요하므로 목적에 덜 적합함.
years = {house['year'] for house in houses}
print("중복 없는 year:", years)
print("정렬해서 보기:", sorted(years))   # 집합은 순서가 없으므로 보기 좋게 정렬

중복 없는 year: {2, 5, 12, 18, 19, 20, 22}
정렬해서 보기: [2, 5, 12, 18, 19, 20, 22]


In [13]:
# ③ 주택 ID로 가격을 조회 → 딕셔너리(dict)
# [선택 이유] 키를 통한 조회: ID(키)로 가격(값)을 바로 찾을 수 있음.
#            ID마다 가격이 하나씩 대응되는 구조와 일치함.
# [선택하지 않은 자료구조: 리스트(list)]
#   - 리스트는 위치(인덱스)로만 접근하므로 ID로 찾으려면 처음부터 반복하며 비교해야 함.
#   - 딕셔너리는 키로 바로 접근하므로 조회가 간단하고 빠름.
price_by_id = {house['ID']: house['price'] for house in houses}
print("ID 3의 가격:", price_by_id.get(3, "ID 3 없음"))

ID 3의 가격: 4.0


In [14]:
# ④ 최소 크기 30, 최대 가격 4를 하나의 묶음으로 저장 → 튜플(tuple)
# [선택 이유] 변경 불가: 생성 후 원소 추가·삭제·교체가 불가능해서
#            검색 기준값이 실수로 바뀌는 것을 막을 수 있음.
#            두 값(최소 크기, 최대 가격)을 하나의 묶음으로 저장함.
# [선택하지 않은 자료구조: 리스트(list)]
#   - 리스트는 변경 가능해서 append, remove, 값 교체가 모두 가능함.
#   - "수정할 수 없어야 한다"는 요구에 맞지 않음.
criteria = (30, 4)   # (최소 크기, 최대 가격)
print("검색 조건:", criteria)

검색 조건: (30, 4)


In [15]:
# 불변 확인: 값을 바꾸려 하면 TypeError 발생
try:
    criteria[0] = 31
except TypeError as e:
    print("변경 불가 확인:", e)

변경 불가 확인: 'tuple' object does not support item assignment
